# 01 — LLM Clients

Demonstrates the Claude client and response cache. Run `00_setup.ipynb` first.

In [ ]:
import sys
sys.path.insert(0, '../../')

import os
import time
from dotenv import load_dotenv
load_dotenv('../../.env')

from src.utils.config_loader import load_config
from src.llm.claude_client import ClaudeClient
from src.llm.cache import ResponseCache
from src.llm.rate_limiter import TokenBucketRateLimiter

## Load Config

In [ ]:
config = load_config('claude')
llm_config = config['llm']
cache_config = config['cache']

print(f"Provider : {llm_config['provider']}")
print(f"Model    : {llm_config['model']}")
print(f"Cache DB : {cache_config['db_path']}")

## Claude Client

Using `claude-haiku-4-5` by default — cheapest model, ideal for high-volume search evaluation loops.
Switch to `claude-sonnet-4-6` in `config/claude.yaml` for higher-quality final evaluation runs.

In [ ]:
client = ClaudeClient(
    model=llm_config['model'],
    api_key=llm_config['api_key'],
)
print(f"ClaudeClient ready: {client.model}")

## Response Cache Demo

First call hits the API. Second identical call (same prompt + model + temperature) returns from the SQLite cache instantly.

In [ ]:
cache = ResponseCache(
    db_path=cache_config['db_path'],
    max_age_hours=cache_config['max_age_hours'],
)

TEST_PROMPT = "What is the capital of France? Answer in one word."
MODEL = llm_config['model']
TEMP = llm_config['temperature']

cached = cache.get(TEST_PROMPT, MODEL, TEMP)
if cached:
    print(f"Cache hit: {cached}")
else:
    print("Cache miss — calling API...")
    start = time.time()
    response = client.generate(TEST_PROMPT, temperature=TEMP)
    elapsed = time.time() - start
    cache.set(TEST_PROMPT, MODEL, TEMP, response)
    print(f"Response ({elapsed:.2f}s): {response}")

    cached = cache.get(TEST_PROMPT, MODEL, TEMP)
    print(f"Second call (cache hit): {cached}")

## Usage Stats

In [ ]:
stats = client.get_usage_stats()
for key, val in stats.items():
    print(f"{key}: {val}")

print("\nCache stats:")
for key, val in cache.stats().items():
    print(f"  {key}: {val}")

## Rate Limiter Demo

In [ ]:
limiter = TokenBucketRateLimiter(requests_per_minute=config['rate_limiter']['requests_per_minute'])

start = time.time()
for i in range(5):
    limiter.acquire()
elapsed = time.time() - start
print(f"5 acquires in {elapsed:.3f}s (all within bucket capacity)")

## Local Model (Optional)

Only runs if `LOCAL_MODEL_PATH` is set and the GGUF file exists.

In [ ]:
from pathlib import Path

local_model_path = os.environ.get('LOCAL_MODEL_PATH', '')

if local_model_path and Path(local_model_path).exists():
    from src.llm.local_client import LocalLlamaClient
    local_client = LocalLlamaClient(model_path=local_model_path)
    response = local_client.generate("What is 2 + 2? Answer briefly.", max_tokens=50)
    print(f"Local model response: {response}")
    print(f"Stats: {local_client.get_usage_stats()}")
else:
    print("Skipping local model — set LOCAL_MODEL_PATH in .env to enable")